In [ ]:
import pandas as pd
from modAL.models import ActiveLearner
from modAL.models import CommitteeRegressor
from modAL.disagreement import vote_entropy_sampling
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import torch
import kan
from kan import KAN, create_dataset_from_data

In [ ]:
import numpy as np

In [ ]:
torch.autograd.set_detect_anomaly(True)

In [ ]:
df = pd.read_csv('../data/processed/train_merge_gen_v4.csv')
for i in df.columns:
    if df[i].dtype is not np.float64:
        df[i] = df[i].astype(np.float64)
print(df.columns)
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
X_pool = train_df.drop("PT_LOSS", axis=1).values
y_pool = train_df["PT_LOSS"].values.reshape(-1, 1)
print(X_pool)
# tX, ty = torch.from_numpy(X_pool).float(), torch.from_numpy(y_pool).float()
# dataset = create_dataset_from_data(tX, ty)
# print(tX, ty)

In [ ]:
class KANWrapper:
    def __init__(self, **params):
        self.model = KAN(**params)

    def fit(self, X, y, opt="LBFGS", steps=80, lamb=0.001):
        dataset = create_dataset_from_data(X, y)
        self.model.fit(dataset, opt=opt, steps=steps, lamb=lamb)
        return self

    def predict(self, X):
        probs = self.model(X)
        return probs

In [ ]:
def qbc(committee, X_sample):
    loss_fn = torch.nn.MSELoss()
    tx = torch.from_numpy(X_sample).float().requires_grad_(True)

    preds = []
    for model in committee.learner_list:
        pred = model.predict(tx)  # (1, 1) или (1,)
        preds.append(pred)

    preds = torch.stack(preds, dim=0)  # (N_models, 1, D) или (N_models, 1)
    f_avg = preds.mean(dim=0)         # (1, D)
    # MSE между всеми предсказаниями и средним
    loss = loss_fn(preds.squeeze(), f_avg.squeeze())
    return loss, tx

In [ ]:
def NA_QBC(committee, X_sample):
    grads = []
    for i in range(X_sample.shape[0]):
        x_s = np.array([X_sample[i]])
        qbc_loss, tx = qbc(committee, x_s)
        if tx.grad is not None:
            tx.grad.zero_()
        qbc_loss.backward(retain_graph=True)
        grads.append(tx.grad.detach().clone())
    return grads

In [ ]:
def NA_query_strategy(comittee, X_sample):
    grads = NA_QBC(comittee, X_sample) # (N, M) -> N
    x_gen = x + alpha * (grad - l_grad_bnd) # из статьи
    steps = get_steps(x)
    x_gen = x + steps * sign(grad - l_grad_bnd)
    for _ in range(grad_iter):
        x_gen = x + steps * sign(grad - l_grad_bnd)
        x = x_gen
    return None, X_generated


# steps = np.array([1, 0.2, ...])

In [ ]:
def get_steps(x):
    if x[0] <= 0.6:
        steps[0] = 0.05

In [ ]:
_, X_generated = committee.query(train_X)
y_new = get_new_y() #
committee.teach(
    X_generated.reshape(1, -1),
    train_y[query_idx].reshape(1, -1),
)

In [ ]:
n_members = 2  # количесво моделей
learner_list = list()
grid1 = [3, 7]
k1 = [5, 3]
n_queries = 10  # Количество итераций активного обучения
# for i in range(n_queries):
for member_idx in range(n_members):
    # initial training data
    n_initial = 7
    train_idx = np.random.choice(range(X_pool.shape[0]), size=n_initial, replace=False)
    X_train = X_pool[train_idx]
    y_train = y_pool[train_idx]
    tX, ty = torch.from_numpy(X_train).float(), torch.from_numpy(y_train).float()
    # creating a reduced copy of the data with the known instances removed
    X_pool = np.delete(X_pool, train_idx, axis=0)
    y_pool = np.delete(y_pool, train_idx)

    # initializing learner
    learner = ActiveLearner(
        estimator=KANWrapper(
            width=[7, 7, 7, 1], grid=grid1[member_idx], k=k1[member_idx], seed=42
        ),  # вот сюда засовываем наш KAN
    )
    learner_list.append(learner)

# assembling the committee

committee = CommitteeRegressor(learner_list=learner_list, query_strategy=qbc)

In [ ]:
grad_ = NA_QBC(committee, X_pool[:2])

In [ ]:
grad_

In [ ]:
print(y_pool[100],type(y_pool[100]))
print(X_pool[100],type(X_pool[100]))
model1 = KANWrapper(width=[7, 7, 7, 1], grid=3, k=5,seed=42)
model1.fit(tX,ty)


In [ ]:
tx1 = torch.Tensor([X_pool[100].tolist()]).float().requires_grad_(True)
ty1 = torch.Tensor([[y_pool[100].item()]]).float()
pred = model1.predict(tx1)

In [ ]:
loss_fn = torch.nn.MSELoss()
loss = loss_fn(pred, ty1)
loss.backward()
# Получаем производные по входным данным
gradients = tx1.grad
print(gradients)

In [ ]:
n_queries = 10  # Количество итераций активного обучения
n_committee = 5  # Количество моделей в комитете
# Основной цикл активного обучения
for i in range(n_queries):
    # Создание комитета моделей
    committee = [RandomForestRegressor() for _ in range(n_committee)]
    
    # Обучение моделей на текущем наборе данных
    for model in committee:
        model.fit(X_train, y_train)
    
    # Получение предсказаний от всех моделей
    predictions = np.array([model.predict(X_pool) for model in committee])

# Вычисление неопределенности (разброс предсказаний)
    uncertainty = np.std(predictions, axis=0)
    
    # Выбор экземпляра с наибольшей неопределенностью
    query_index = np.argmax(uncertainty)
    
    # Добавление выбранного экземпляра в обучающую выборку
    X_train = np.vstack((X_train, X_pool[query_index].reshape(1, -1)))
    y_train = np.append(y_train, y_pool[query_index])
    
    # Удаление выбранного экземпляра из пула
    X_pool = np.delete(X_pool, query_index, axis=0)
    y_pool = np.delete(y_pool, query_index)

In [ ]:
# def qbc(committee, X_pool, y_pool, n_initial):
#     qsum = [0 for i in range(len(n_initial)]
#     f_average = [0 for i in range(len(n_initial)]
#     train_idx = np.random.choice(range(X_pool.shape[0]), size=n_initial, replace=False)
#     Sbp = 2
#     X_add = X_pool[train_idx]
#     y_add = y_pool[train_idx]
#     for i in range(n_initial):
#         sample = X_add[i]
#         tx = torch.from_numpy(sample).float() 
#         for model_ind in range(len(committee)):
#             f_average[n_initial] += committee[model_ind].predict(tx)
#         f_average[n_initial] = f_average[n_initial]/len(committee)
#     for i in range(n_initial):
#         sample = X_add[i]
#         for model in committee:
#             qsum[i] += (model.predict(tx) - f_average[i])**2
#         qsum[i] /= len(committee)
#     return qsum